# M10.2 · Thread'ler ve eşzamanlama — yarışlar, atomikler, kilitler, koşul değişkenleri

Süreçten (fork, ayrı adres uzayı) thread'e (yalnız yığını ayrı, belleği paylaşan hafif yürütme birimi); POSIX thread'lerinden C++11 `std::thread`'e: oluşturma, argüman geçirme (değerle/`std::ref`), `join`/`detach`, ömür. Paylaşılan veri: veri yarışı (data race, C/C++'ta tanımsız davranış) ve yarış koşulu (race condition), banka hesabı örneği, oku–değiştir–yaz atomikliği, `std::atomic`, `std::mutex`/`lock_guard`, koşul değişkenleriyle sınırlı tampon, kilitlenme (deadlock) ve kilit sıralaması, sahte paylaşım (false sharing). Saf Python'da tüm iç içe geçmeleri (interleaving) gezen bir doğrulayıcı, NumPy'da yarış olasılığı, C++'ta atomik/mutex/özel histogram, sahte paylaşım ve üretici–tüketici, sonda kendi thread havuzumuz ve görev ayrıntı düzeyi.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-2-threads-sync.html](https://m-gonul.github.io/pytorch-lab/m10-2-threads-sync.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_2_threads_sync.py
CHECKS = {
 "b": "from labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    inc = [(\"load\", \"x\", \"r\"), (\"add\", \"r\", 1), (\"store\", \"x\", \"r\")]\n    locked = [(\"lock\", \"m\")] + inc + [(\"unlock\", \"m\")]\n\n    def race():\n        out, dl = need(ns, \"explore\")([inc, inc], {\"x\": 0})\n        ensure(out == {((\"x\", 1),), ((\"x\", 2),)} and not dl, \"kayıp güncelleme mümkün\")\n        out, dl = ns[\"explore\"]([locked, locked, locked], {\"x\": 0})\n        ensure(out == {((\"x\", 3),)} and not dl, \"kilitle yalnız doğru sonuç\")\n        out3, _ = ns[\"explore\"]([inc, inc, inc], {\"x\": 0})\n        ensure({v for ((_, v),) in out3} == {1, 2, 3}, \"3 thread: 1, 2, 3\")\n        dep = lambda k: [(\"load\", \"b\", \"r\"), (\"add\", \"r\", k), (\"store\", \"b\", \"r\")]\n        outb, _ = ns[\"explore\"]([dep(1000), dep(500)], {\"b\": 10000})\n        ensure({v for ((_, v),) in outb} == {11500, 11000, 10500}, \"banka örneği\")\n        ensure(need(ns, \"count_interleavings\")([2, 2]) == 6 and ns[\"count_interleavings\"]([3, 3]) == 20 and ns[\"count_interleavings\"]([3, 3, 3]) == 1680, \"multinom\")\n    R_.test(\"yarışlar\", race)\n\n    def dead():\n        t1 = [(\"lock\", \"A\"), (\"lock\", \"B\"), (\"load\", \"x\", \"r\"), (\"add\", \"r\", 1), (\"store\", \"x\", \"r\"), (\"unlock\", \"B\"), (\"unlock\", \"A\")]\n        t2 = [(\"lock\", \"B\"), (\"lock\", \"A\"), (\"load\", \"x\", \"r\"), (\"add\", \"r\", 1), (\"store\", \"x\", \"r\"), (\"unlock\", \"A\"), (\"unlock\", \"B\")]\n        out, dl = ns[\"explore\"]([t1, t2], {\"x\": 0})\n        ensure(dl and out == {((\"x\", 2),)}, \"ters sırada kilit: kilitlenme mümkün ama biten yollar doğru\")\n        out, dl = ns[\"explore\"]([t1, t1], {\"x\": 0})\n        ensure(not dl, \"aynı sırada kilit: kilitlenme yok\")\n        R_.table(\"Model denetimi\", [\"program\", \"olası sonuçlar\", \"kilitlenme\"],\n                 [[\"2 × (oku, +1, yaz)\", \"x ∈ {1, 2}\", \"yok\"], [\"3 × kilitli artış\", \"x = 3\", \"yok\"], [\"A→B ve B→A kilit sırası\", \"x = 2 (bitenlerde)\", \"VAR\"], [\"A→B ve A→B\", \"x = 2\", \"yok\"]],\n                 note=\"Test çalıştırmak zamanlamalardan birini görür; durum uzayını gezmek hepsini. Kilitlenme yalnız belirli iç içe geçmelerde çıkar — bu yüzden testlerde nadiren yakalanır.\")\n    R_.test(\"kilitlenme\", dead)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: (max_loops(src, 0, func=\"lost_update_prob\"), max_loops(src, 0, func=\"expected_lost\")))\n\n    def t():\n        rows = []\n        for L in (0.001, 0.01, 0.05, 0.2):\n            e = need(ns, \"lost_update_prob\")(L, 1.0, 400000, 1)\n            a = 1 - (1 - L) ** 2\n            close(e, a, atol=0.003, what=f\"L = {L}\")\n            rows.append([L, f\"{e:.4f}\", f\"{a:.4f}\"])\n        close(need(ns, \"expected_lost\")(2, 0.1, 1.0, 400000, 2), 1 - 0.81, atol=0.003, what=\"k = 2 ile tutarlı\")\n        k_rows = [[k, f\"{ns['expected_lost'](k, 0.001, 1.0, 20000, 3):.3f}\"] for k in (2, 8, 32, 128)]\n        ensure(float(k_rows[-1][1]) > float(k_rows[0][1]) * 20, \"kayıp thread sayısıyla süperdoğrusal\")\n        R_.table(\"İki thread, kritik pencere L (çalışma süresi T = 1)\", [\"L\", \"deney\", \"1 − (1 − L/T)²\"], rows,\n                 note=\"Pencere küçükse yarış nadirdir — testlerden kaçar ama milyonlarca işlemde kesin olur. Aşağıda: k thread, L = 0.001: \" + \", \".join(f\"k = {a}: {b}\" for a, b in k_rows) + \" kayıp/çalıştırma (~k²L).\")\n    R_.test(\"yarış olasılığı\", t)\n",
 "d": "import os\nimport time\nimport torch\nfrom labcheck import need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    ext = need(ns, \"build\")()\n    P = max(2, min(8, os.cpu_count() or 2))\n\n    def hist():\n        g = torch.Generator().manual_seed(0)\n        img = (torch.randn(2048, 2048, generator=g) * 30 + 128).clamp(0, 255).to(torch.uint8)\n        ref = torch.bincount(img.flatten(), minlength=256)\n        rows = []\n        for mode in (\"single\", \"atomic\", \"mutex\", \"private\"):\n            out = ext.histogram(img, P, mode)\n            ensure(torch.equal(out, ref), f\"{mode}: histogram doğru olmalı\")\n            best = 1e9\n            for _ in range(2 if mode == \"mutex\" else 3):\n                t0 = time.perf_counter()\n                ext.histogram(img, P, mode)\n                best = min(best, time.perf_counter() - t0)\n            rows.append([mode, fmt_time(best), \"doğru\"])\n        flat = torch.full((2048, 2048), 7, dtype=torch.uint8)\n        lost = []\n        for _ in range(3):\n            lost.append(int(flat.numel() - ext.histogram(flat, P, \"racy\")[7]))\n        rows.append([\"racy (tek değerli görüntü)\", \"\", f\"kayıp güncelleme: {', '.join(map(str, lost))}\"])\n        R_.table(f\"256 kutulu histogram, 2048×2048 piksel, {P} thread\", [\"yöntem\", \"süre\", \"sonuç\"], rows,\n                 note=\"Atomik ve mutex doğru ama her piksel paylaşılan bir konuma dokunur: kutular az, çekişme çok. Thread'e özel histogram (PCAM: birleştirme) iletişimi 256 · p toplamaya indirir. 'racy' C++ açısından tanımlı (atomik oku, atomik yaz) ama yarış koşulu içerir; çok çekirdekte kayıp neredeyse kesindir.\")\n    R_.test(\"histogram\", hist)\n\n    def fs():\n        rows = []\n        for padded in (False, True):\n            best = 1e9\n            for _ in range(3):\n                dt, cnt = ext.false_sharing(P, 2_000_000, padded)\n                best = min(best, dt)\n                ensure(torch.equal(cnt, torch.full((P,), 2_000_000, dtype=torch.int64)), \"sayaçlar doğru\")\n            rows.append([\"alignas(64) dolgulu\" if padded else \"bitişik dizi\", fmt_time(best)])\n        R_.table(f\"{P} thread × 2 M fetch_add, her thread kendi sayacına\", [\"yerleşim\", \"süre\"], rows,\n                 note=\"Mantıksal olarak paylaşım yok, ama donanım için aynı önbellek satırı paylaşılıyor: satır çekirdekler arasında 'ping-pong' yapar.\")\n    R_.test(\"sahte paylaşım\", fs)\n\n    def bb():\n        for (n, cap, pr, co) in ((10000, 4, 1, 1), (20000, 8, 3, 2), (5000, 1, 4, 4)):\n            s, occ = ext.bounded_buffer(n, cap, pr, co)\n            ensure(s == n * (n + 1) // 2, f\"tüm öğeler bir kez tüketilmeli (n = {n})\")\n            ensure(1 <= occ <= cap, \"doluluk kapasiteyi aşmamalı\")\n    R_.test(\"sınırlı tampon\", bb)\n",
 "e": "import os\nimport torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    ext = need(ns, \"build\")()\n    P = max(2, min(8, os.cpu_count() or 2))\n\n    def t():\n        g = torch.Generator().manual_seed(0)\n        x = torch.rand(10_000_000, generator=g, dtype=torch.float64)\n        ref = float(x.sum())\n        for ch in (1000, 123457, 10_000_000):\n            close(ext.pool_sum(x, P, ch), ref, rtol=1e-12, what=f\"havuz toplamı (chunk {ch})\")\n        close(ext.spawn_sum(x, P), ref, rtol=1e-12, what=\"spawn toplamı\")\n        rows = []\n        for ch in (100, 1000, 10_000, 100_000, 1_000_000):\n            tt = bench(lambda: ext.pool_sum(x, P, ch), repeat=3, number=1)\n            rows.append([f\"havuz, chunk {ch:,}\".replace(\",\", \" \"), f\"{(x.numel() + ch - 1) // ch:,}\".replace(\",\", \" \"), fmt_time(tt), fmt_time(tt / ((x.numel() + ch - 1) // ch))])\n        rows.append([f\"{P} thread aç-kapat (eşit bölme)\", P, fmt_time(bench(lambda: ext.spawn_sum(x, P), repeat=3, number=1)), \"\"])\n        old = torch.get_num_threads()\n        for nt in sorted({1, P}):\n            torch.set_num_threads(nt)\n            rows.append([f\"torch.sum, {nt} iç thread\", \"\", fmt_time(bench(lambda: x.sum(), repeat=5, number=1)), \"\"])\n        torch.set_num_threads(old)\n        R_.table(f\"10 M float64 toplamı, {P} işçi\", [\"yöntem\", \"görev\", \"süre\", \"görev başına\"], rows,\n                 note=\"Çok küçük görevlerde süreyi kuyruk kilidi, std::function tahsisi ve uyandırmalar belirler; çok büyük görevlerde paralellik kaybolur (tek görev = tek thread). Pratik kural: görev başına en az ~10–100 µs iş.\")\n    R_.test(\"thread havuzu ve ayrıntı düzeyi\", t)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for n in (2, 3, 4, 5, 6):\n            d0, s0 = need(ns, \"philosophers\")(n, False)\n            d1, s1 = ns[\"philosophers\"](n, True)\n            ensure(d0 and not d1, f\"n = {n}: saf sırada kilitlenme var, sıralı almada yok\")\n            rows.append([n, s0, \"VAR\", s1, \"yok\"])\n        R_.table(\"Yemek yiyen filozoflar (tek tur), tüm iç içe geçmeler\", [\"n\", \"durum (sol→sağ)\", \"kilitlenme\", \"durum (küçük çatal önce)\", \"kilitlenme\"], rows,\n                 note=\"Herkes sol çatalını alınca döngüsel bekleme oluşur. Küresel bir kaynak sırası döngüyü imkânsız kılar: en büyük numaralı çatalı bekleyen hep ilerleyebilir.\")\n    R_.test(\"filozoflar\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        res = {}\n        for v in (\"correct\", \"turn_first\", \"no_turn\"):\n            viol, dl, n = need(ns, \"peterson_states\")(v)\n            res[v] = (viol, dl)\n            rows.append([v, n, \"VAR\" if viol else \"yok\", \"VAR\" if dl else \"yok\"])\n        ensure(res[\"correct\"] == (False, False), \"Peterson: dışlama korunur, kilitlenme yok\")\n        ensure(res[\"turn_first\"][0], \"önce turn yazılırsa dışlama bozulur\")\n        ensure(res[\"no_turn\"][1] and not res[\"no_turn\"][0], \"yalnız bayraklar: güvenli ama kilitlenebilir\")\n        R_.table(\"Peterson algoritması ve varyantları, tam durum uzayı\", [\"varyant\", \"durum\", \"ikisi birden kritik bölgede\", \"kilitlenme (ikisi de sonsuza dek bekler)\"], rows,\n                 note=\"Kanıt sıralı tutarlılık (her adım atomik ve program sırasında) varsayar. Gerçek x86'da depolama tamponu (store buffer) bayrak yazısını geciktirir; bu yüzden C++'ta seq_cst atomikler ya da çit (fence) gerekir — ödevin konusu.\")\n    R_.test(\"Peterson\", t)\n",
 "x3": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        res = {}\n        for pol in (\"reader\", \"writer\"):\n            ww, wr, dw = need(ns, \"simulate\")(pol, 0.6, 0.05, 20000, 1)\n            res[pol] = (ww, wr, dw)\n            rows.append([pol, f\"{ww:.1f}\", f\"{wr:.1f}\", dw])\n        ensure(res[\"reader\"][0] > 5 * res[\"writer\"][0], \"okuyucu önceliğinde yazıcılar aç kalmalı\")\n        ensure(res[\"writer\"][1] > res[\"reader\"][1], \"yazıcı önceliğinde okuyucular daha çok bekler\")\n        R_.table(\"Okuyucu–yazıcı kilidi: p_okuyucu = 0.6, p_yazıcı = 0.05 / tik, 20 000 tik\", [\"öncelik\", \"ort. yazıcı beklemesi\", \"ort. okuyucu beklemesi\", \"biten yazıcı\"], rows,\n                 note=\"Sürekli okuyucu akışı varken okuyucu öncelikli kilit yazıcıya hiç boşluk bırakmaz (açlık). Adil kilitler gelen sırasına göre (FIFO) davranarak ikisinin ortasında durur; C++17 std::shared_mutex'in politikası tanımsızdır.\")\n    R_.test(\"okuyucu–yazıcı\", t)\n",
 "x4": "from collections import deque, Counter\nfrom labcheck import need, ensure\n\n\nclass _Buf(deque):\n    def __init__(self, cap):\n        super().__init__()\n        self.cap, self.viol = cap, 0\n\n    def append(self, x):\n        super().append(x)\n        if len(self) > self.cap:\n            self.viol += 1\n\n\ndef check(ns, R_):\n    Sem, run = ns[\"Sem\"], ns[\"run\"]\n\n    def t_sem():\n        s = Sem(1)\n        ensure(need(ns, \"sem_P\")(s, 0) is True and s.n == 0, \"P: sayaç > 0 ise azalt ve devam\")\n        ensure(ns[\"sem_P\"](s, 1) is False and list(s.q) == [1], \"P: sayaç 0 ise kuyruğa gir, bloklan\")\n        ns[\"sem_P\"](s, 2)\n        ensure(need(ns, \"sem_V\")(s) == 1 and s.n == 0, \"V: bekleyen varsa ilkini (FIFO) uyandır, sayaç artmaz\")\n        ensure(ns[\"sem_V\"](s) == 2 and ns[\"sem_V\"](s) is None and s.n == 1, \"V: bekleyen yoksa sayaç artar\")\n    R_.test(\"semafor işlemleri\", t_sem)\n\n    def system(cap, per, prod, pills=True, seed=0):\n        buf, out = _Buf(cap), []\n        e, f, m, d = Sem(cap), Sem(0), Sem(1), Sem(0)\n        items = [list(range(k * 100, k * 100 + per)) for k in range(3)]\n        th = [prod(buf, e, f, m, it, d) for it in items] + [need(ns, \"consumer\")(buf, e, f, m, out) for _ in range(2)]\n        if pills:\n            th.append(need(ns, \"closer\")(buf, e, f, m, d, 3, 2))\n        r = run(th, seed)\n        return r, sorted(out) == sorted(sum(items, [])), buf.viol\n\n    def bad(buf, empty, full, mutex, items, done):\n        for x in items:\n            yield (\"P\", mutex)\n            yield (\"P\", empty)\n            yield (\"do\", lambda x=x: buf.append(x))\n            yield (\"V\", mutex)\n            yield (\"V\", full)\n        yield (\"V\", done)\n\n    def t_sys():\n        prod = need(ns, \"producer\")\n        for s in range(150):\n            r, same, viol = system(2, 6, prod, seed=s)\n            ensure(r == \"ok\", f\"tohum {s}: sistem {r} ile bitti (hepsi sonlanmalı)\")\n            ensure(same, f\"tohum {s}: her öğe tam bir kez tüketilmeli\")\n            ensure(viol == 0, f\"tohum {s}: tampon kapasiteyi aştı\")\n        rows = []\n        for cap, per in ((1, 1), (4, 1), (1, 6), (4, 6)):\n            a = Counter(system(cap, per, prod, seed=s)[0] for s in range(200))\n            b = Counter(system(cap, per, bad, seed=s)[0] for s in range(200))\n            c = Counter(system(cap, per, prod, pills=False, seed=s)[0] for s in range(200))\n            rows.append([cap, 3 * per, f\"{a['deadlock'] / 200:.2f}\", f\"{b['deadlock'] / 200:.2f}\", f\"{c['deadlock'] / 200:.2f}\"])\n        ensure(float(rows[2][3]) > 0.5 and float(rows[0][2]) == 0, \"ters sıra kilitlenir, doğru sıra kilitlenmez\")\n        R_.table(\"Rastgele zamanlamalarda kilitlenme oranı (200 tohum, 3 üretici, 2 tüketici)\", [\"kapasite\", \"toplam öğe\", \"doğru: P(empty) → P(mutex)\", \"ters: P(mutex) → P(empty)\", \"zehirli hap yok\"], rows,\n                 note=\"Ters sırada üretici mutex'i tutarken tamponun boşalmasını bekler; tamponu boşaltacak tüketici de o mutex'i bekler: döngüsel bekleme. Tampon hiç dolmuyorsa (öğe ≤ kapasite) hata gizli kalır — testte görünmeyen, üretimde ortaya çıkan türden. Zehirli hap olmadan tüketiciler 'full' semaforunda sonsuza dek bekler: sonlandırma da bir eşzamanlama problemidir.\")\n    R_.test(\"sınırlı tampon ve sonlandırma\", t_sys)\n",
 "x5": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def trial(tagged, seed, W=3, rounds=4, nodes=(\"A\", \"B\", \"C\")):\n        mem = {\"top\": (None, 0), \"next\": {n: None for n in nodes}, \"retries\": 0}\n        for n in reversed(nodes):\n            mem[\"next\"][n] = mem[\"top\"][0]\n            mem[\"top\"] = (n, 0)\n        steps = ns[\"schedule\"]([ns[\"worker\"](mem, rounds, tagged) for _ in range(W)], seed)\n        return ns[\"stack_ok\"](mem, list(nodes)), mem[\"retries\"], steps\n\n    def t_cas():\n        mem = {\"x\": (1, 0)}\n        ensure(need(ns, \"cas\")(mem, \"x\", (1, 0), (2, 1)) is True and mem[\"x\"] == (2, 1), \"CAS: beklenen eşitse yaz, True\")\n        ensure(ns[\"cas\"](mem, \"x\", (1, 0), (3, 3)) is False and mem[\"x\"] == (2, 1), \"CAS: eşit değilse dokunma, False\")\n        mem = {\"top\": (None, 0), \"next\": {\"A\": None, \"B\": None}, \"retries\": 0}\n        for g in (need(ns, \"push\")(mem, \"A\", True), ns[\"push\"](mem, \"B\", True)):\n            for _ in g:\n                pass\n        g = need(ns, \"pop\")(mem, True)\n        try:\n            while True:\n                next(g)\n        except StopIteration as e:\n            ensure(e.value == \"B\" and mem[\"top\"][0] == \"A\", \"tek thread: LIFO\")\n    R_.test(\"CAS, push, pop (tek thread)\", t_cas)\n\n    def t_aba():\n        rows, res = [], {}\n        for tagged in (False, True):\n            for W in (2, 3, 4):\n                r = [trial(tagged, s, W=W) for s in range(600)]\n                bad = sum(not ok for ok, _, _ in r) / len(r)\n                res[(tagged, W)] = bad\n                rows.append([\"etiketli (ptr, sayaç)\" if tagged else \"yalın işaretçi\", W, f\"{bad:.3f}\", f\"{sum(x for _, x, _ in r) / len(r) / (W * 4 * 2):.2f}\"])\n        ensure(res[(False, 3)] > 0 and all(res[(True, w)] == 0 for w in (2, 3, 4)), f\"ABA yalın CAS'ta görülmeli, etiketlide asla {res}\")\n        R_.table(\"Treiber yığını: 3 düğüm, her işçi 4 kez pop + aynı düğümü push (600 rastgele zamanlama)\", [\"CAS karşılaştırması\", \"işçi\", \"bozulan yığın oranı\", \"işlem başına yeniden deneme\"], rows,\n                 note=\"ABA: thread 1 top = A ve A.next = B'yi okur, kesilir; thread 2 A'yı ve B'yi çıkarır, A'yı geri koyar. top yine A olduğu için thread 1'in CAS'ı başarılı olur ve top'u artık yığında olmayan B'ye çevirir. Sürüm sayacı (etiket) her başarılı CAS'ta arttığı için 'aynı görünen' durum ayırt edilir (çift genişlikli CAS ya da LL/SC ile). Kilitsiz algoritma sistem genelinde ilerleme garanti eder ama tek bir thread'in sürekli yeniden denemesini (açlık) engellemez.\")\n    R_.test(\"ABA sorunu\", t_aba)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_2_threads_sync as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Süreç ve thread:** OS süreçleri zaman dilimleriyle döndürür (bağlam değişimi: zamanlayıcı kesmesi, G/Ç, sistem çağrısı). `fork()` yeni süreç yaratır: ayrı adres uzayı, iletişim için boru/paylaşımlı bellek — pahalı. **Thread** yalnız gerekeni (yığın, yazmaçlar) çoğaltır; kod, yığıt (heap) ve küresel değişkenler paylaşılır. Kullanım: performans (çekirdekler), arka plan işleri, asenkron işleme, daha iyi program yapısı.

**C++11 thread'leri** (pThreads modelini standartlaştırır): `std::thread t(f, args…)` hemen başlar; `t.join()` bekler, `detach()` bağımsız bırakır (ana thread bitince süreç biter). Argümanlar **kopyalanır**; referans için `std::ref`. Bir thread nesnesi join edilmeden yok edilirse program sonlanır (`std::terminate`).

**Paylaşılan veri:** salt okumalar güvenlidir. Bir thread okurken diğeri yazarsa? Banka hesabı: bakiye 10 000, maaş 1000 ve 500 aynı anda yatırılıyor; her iki thread "oku → topla → yaz" yaparsa sonuç 11 500, 11 000 ya da 10 500 olabilir.

- **Veri yarışı:** iki thread aynı konuma eşzamanlanmadan erişir ve en az biri yazar → C/C++'ta **tanımsız davranış** (8 bayttan büyük nesneler yırtılabilir, derleyici değeri yazmaçta tutabilir).
- **Yarış koşulu:** doğruluk zamanlamaya bağlı. Veri yarışı olmadan da olur (atomik oku + atomik yaz, arada başkası yazabilir).
- **Atomik:** `std::atomic<T>` tek işlemde oku–değiştir–yaz (`fetch_add`), sayaçlar ve biriktiriciler için ideal; yazma diğer thread'lerce görünür.
- **Karşılıklı dışlama:** `std::mutex` + `std::lock_guard`/`unique_lock` ile **kritik bölge**; çekişme yüksekse yavaş. Çoğu zaman en iyisi: **thread'e özel kopya** + sonda birleştirme.
- **Koşul değişkeni:** `cv.wait(lock, pred)` kilidi bırakıp uyur; `notify_one/all`. Sahte uyanmaya karşı yüklemle.
- **Kilitlenme (Coffman):** karşılıklı dışlama, tut-ve-bekle, zorla alınamama, döngüsel bekleme. Önlem: tüm kilitleri **aynı sırada** al (ya da `std::scoped_lock` ile birden).
- **Sahte paylaşım:** farklı thread'lerin farklı değişkenleri aynı 64 baytlık önbellek satırındaysa her yazma satırı diğer çekirdeklerde geçersizler → `alignas(64)`.

:::ayrıntı Bellek tutarlılığı: sıralı tutarlılık ve doğrusallaştırılabilirlik
**Sıralı tutarlılık** (Lamport 1979): bir çok thread'li çalıştırmanın sonucu, tüm thread'lerin işlemlerinin *bir* serpiştirilmesiyle (her thread kendi program sırasında) elde edilebilen sonuçlardan biridir. B aşamasındaki doğrulayıcımız tam olarak bu modeli varsayar. Gerçek donanım daha zayıftır: x86 TSO'da bir thread'in yazması kendi depolama tamponunda bekleyebilir, bu yüzden Peterson algoritması (E2) bellek bariyeri olmadan bozulur. C++'ta varsayılan `std::atomic` işlemleri (`memory_order_seq_cst`) sıralı tutarlılık sağlar; `relaxed`/`acquire`/`release` daha ucuz ama daha zayıftır.

**Doğrusallaştırılabilirlik** (Herlihy & Wing 1990): eşzamanlı bir nesnenin (kuyruk, yığın, sayaç) her işlemi, çağrısı ile dönüşü arasında bir anda "atomik olarak gerçekleşmiş" gibi görünür ve bu anlar gerçek zaman sırasına uyar. Sıralı tutarlılıktan güçlüdür (gerçek zaman sırasını da korur) ve **birleştirilebilir**dir: her nesne ayrı ayrı doğrusallaştırılabilirse bütün sistem de öyledir. E5'teki kilitsiz yığının doğrusallaştırma noktası başarılı CAS anıdır.
:::

:::ayrıntı Kullanıcı düzeyi thread'ler ve fiber'ler
İşletim sistemi thread'i çekirdek tarafından zamanlanır: her bağlam değişimi bir sistem çağrısı ve önbellek kaybı demektir (mikrosaniyeler). **Kullanıcı düzeyi thread / fiber / eşyordam** (coroutine) ise kullanıcı alanında, işbirlikçi olarak (kendi isteğiyle `yield` ederek) değişir: oluşturma ve geçiş çok ucuzdur, yüz binlercesi tutulabilir. Bedeli: bir fiber bloklayan bir sistem çağrısı yaparsa (dosya okuma) onu taşıyan çekirdek thread'i ve dolayısıyla üzerindeki tüm fiber'ler durur; çok çekirdeği kullanmak için M:N eşleme (Go'nun goroutine'leri, Windows fiber'leri, C++20 eşyordamları + iş parçacığı havuzu) gerekir. E4 ve E5'teki zamanlayıcımız tam olarak bir fiber zamanlayıcısıdır: thread'ler `yield` ettikleri yerde kesilir.
:::

:::uyarı Çok thread'li kodda hata ayıklama
Yarış hataları zamanlamaya bağlıdır: hata ayıklayıcı ya da `print` zamanlamayı değiştirip hatayı gizleyebilir ("Heisenbug"). Yararlı alışkanlıklar: paylaşılan durumu en aza indir (thread'e özel kopya + birleştirme); kilitleri her zaman aynı sırada ve RAII ile al (`lock_guard`, `scoped_lock`); bekleme her zaman koşul döngüsünde; testleri çok kez ve farklı yüklerle çalıştır; dinamik analiz araçları kullan (ThreadSanitizer: `-fsanitize=thread`, Helgrind). E4/E5'teki gibi rastgele zamanlama keşfi, gerçek araçların (ör. CHESS, Loom) basit bir hâlidir.
:::

## 1 · Elle

<!-- çözüm -->

LOAD₁ LOAD₂ STORE… gibi 4 olaydan ikisi T1'in: $\binom42=6$ sıralama. Yalnız T1 tamamen önce ya da T2 tamamen önce (2 sıra) iki artışı korur; diğer 4'ünde iki thread de 0 okur → x = 1.

Banka: ikisi de 10 000 okursa son yazan kazanır → 11 000 ya da 10 500; sıralıysa 11 500.

Filozoflar: her yiyen iki çatal kullanır, $\lfloor5/2\rfloor=2$. Özel histogram: her thread 256 kutuyu ana histograma ekler: $4\cdot256$ toplama — piksel sayısından bağımsız.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki thread, her biri x = x + 1'i iki adımda yapar: LOAD, STORE(+1); x = 0**

- `il`: Olası iç içe geçme (interleaving) sayısı
- `lost`: Bunlardan kaç tanesi x = 1 bırakır (kayıp güncelleme)?

**Banka: bakiye 10 000, eşzamanlı 1000 ve 500 yatırma (oku → topla → yaz)**

- `bank`: Olası farklı son bakiye sayısı

**Diğer**

- `phil`: 5 filozof, 5 çatal: aynı anda en çok kaç kişi yiyebilir?
- `cl`: 64 baytlık önbellek satırına sığan int64 sayaç
- `merge`: 4 thread'e özel 256 kutulu histogramın birleştirilmesi: toplama sayısı

In [ ]:
cevaplar = dict(
    il=None,
    lost=None,
    bank=None,
    phil=None,
    cl=None,
    merge=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"il":[6,0],"lost":[4,0],"bank":[3,0],"phil":[2,0],"cl":[8,0],"merge":[1024,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**İç içe geçme doğrulayıcı** (saf Python, model denetimi). Bir thread komut listesidir:

- `("load", var, reg)`, `("add", reg, k)`, `("store", var, reg)` — yazmaçlar thread'e özel;
- `("lock", m)`, `("unlock", m)` — kilit başkasındaysa thread ilerleyemez.

- `explore(threads, init)` → `(sonuçlar, kilitlenme_var_mı)`: tüm erişilebilir durumları DFS ile gez (durum = program sayaçları, yazmaçlar, paylaşılan değişkenler, kilit sahipleri; ziyaret edilenleri hatırla). Bütün thread'ler bittiğinde paylaşılan değişkenlerin son hâlini (sıralı `tuple` öğeleri) kümeye ekle; hiçbir thread ilerleyemiyor ama hepsi bitmemişse kilitlenme.
- `count_interleavings(lengths)` — kilitsiz programların toplam sıralama sayısı (multinom katsayısı).

In [ ]:
%%writefile stage_b.py
def explore(threads, init):
    ...


def count_interleavings(lengths):
    ...


inc = [("load", "x", "r"), ("add", "r", 1), ("store", "x", "r")]
print(explore([inc, inc], {"x": 0}))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Yarış olasılığı**, NumPy, döngüsüz. İki thread aynı sayaca oku–değiştir–yaz yapar; her birinin başlangıcı $[0,T]$'de düzgün, okuma–yazma arası L. Aralıklar örtüşürse güncelleme kaybolur.

- `lost_update_prob(L, T, n_trials, seed=0)` → deneysel olasılık; analitik $1-(1-L/T)^2$ ile karşılaştır.
- `expected_lost(k, L, T, n_trials, seed=0)` — k thread: her biri tek artış; sonuç = örtüşme bileşeni başına 1 (bir bileşendeki tüm artışlardan yalnız biri kalır… ). Basitleştirme: başlangıçları sırala; ardışık başlangıç farkı < L ise aynı bileşen. Kayıp = k − bileşen sayısı; ortalama.

In [ ]:
%%writefile stage_c.py
import numpy as np


def lost_update_prob(L, T, n_trials, seed=0):
    ...


def expected_lost(k, L, T, n_trials, seed=0):
    ...


print(lost_update_prob(0.1, 1.0, 100000), 1 - 0.9 ** 2)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

C++ (`load_inline`, `std::thread`).

- `histogram(img, nthreads, mode)` — `img` uint8 bitişik; 256 kutulu int64 histogram. `mode`:
  - `"single"` — tek thread;
  - `"atomic"` — `std::atomic<int64_t> bins[256]`, `fetch_add(1, relaxed)`;
  - `"mutex"` — tek `std::mutex`, her piksel için kilitle;
  - `"private"` — her thread kendi `int64_t[256]`'sı, sonda birleştir;
  - `"racy"` — **bilerek yanlış**: `v = bins[x].load(); bins[x].store(v + 1);` (veri yarışı yok — atomik — ama yarış koşulu var).
- `false_sharing(nthreads, iters, padded)` → `(geçen_saniye, sayaçlar)`: her thread kendi sayacını `iters` kez `fetch_add`; `padded` ise sayaçlar `alignas(64)` yapılarda.
- `bounded_buffer(n_items, cap, n_prod, n_cons)` → `(tüketilenlerin_toplamı, gözlenen_en_büyük_doluluk)`: üreticiler 1..n_items sayılarını paylaşır, tüketiciler `std::condition_variable` ile sınırlı kuyruktan çeker; bitişte "zehir hapı" (−1) her tüketiciye.

In [ ]:
%%writefile stage_d.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>
#include <thread>
#include <atomic>
#include <mutex>
#include <condition_variable>
#include <queue>
#include <vector>
#include <chrono>
#include <tuple>

torch::Tensor histogram(torch::Tensor img, int64_t nthreads, std::string mode) {
    // TODO
    return torch::zeros({256}, torch::kInt64);
}

std::tuple<double, torch::Tensor> false_sharing(int64_t nthreads, int64_t iters, bool padded) {
    // TODO
    return {0.0, torch::zeros({nthreads}, torch::kInt64)};
}

std::tuple<int64_t, int64_t> bounded_buffer(int64_t n_items, int64_t cap, int64_t n_prod, int64_t n_cons) {
    // TODO
    return {0, 0};
}
"""


def build():
    name = "ptl_m102_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["histogram", "false_sharing", "bounded_buffer"], extra_cflags=["-O2"], verbose=False)


ext = build()
img = torch.randint(0, 256, (256, 256), dtype=torch.uint8)
print((ext.histogram(img, 4, "private") - torch.bincount(img.flatten(), minlength=256)).abs().sum())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Kendi thread havuzumuz** (C++): sabit işçi thread'leri, `std::queue<std::function<void()>>`, mutex + koşul değişkeni.

- `pool_sum(x, nthreads, chunk)` — `x` float64; her `chunk` elemanlık parça için bir görev, kısmi toplamlar diziye, ana thread bitişi bekler (atomik sayaç + cv). Toplam (double).
- `spawn_sum(x, nthreads)` — havuzsuz: nthreads adet thread aç, eşit böl.

Kontrol, görev ayrıntı düzeyini (chunk) tarar ve PyTorch'un kendi iç paralelliğiyle (`torch.set_num_threads`) karşılaştırır.

In [ ]:
%%writefile stage_e.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>
#include <thread>
#include <mutex>
#include <condition_variable>
#include <functional>
#include <queue>
#include <vector>
#include <atomic>

double pool_sum(torch::Tensor x, int64_t nthreads, int64_t chunk) {
    // TODO: thread havuzu
    return 0.0;
}

double spawn_sum(torch::Tensor x, int64_t nthreads) {
    // TODO
    return 0.0;
}
"""


def build():
    name = "ptl_m102e_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["pool_sum", "spawn_sum"], extra_cflags=["-O2"], verbose=False)


ext = build()
x = torch.rand(100000, dtype=torch.float64)
print(ext.pool_sum(x, 4, 1000) - float(x.sum()))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Yemek yiyen filozoflar: kilitlenme ve kaynak sıralaması

- `philosophers(n, ordered)` → `(kilitlenme_var_mı, erişilebilir_durum_sayısı)`: n filozof, çatal i ve (i+1) mod n; her filozof sol, sağ çatalı alır, yer, ikisini bırakır, biter (tek tur). `ordered=True`: önce küçük numaralı çatal. Tüm iç içe geçmeleri durum uzayında gez.

In [ ]:
%%writefile extra_x1.py
def philosophers(n, ordered):
    ...


print(philosophers(3, False), philosophers(3, True))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Peterson algoritması: karşılıklı dışlamanın kanıtı (durum uzayı)

**Peterson** (iki thread): `flag[i] = 1; turn = j; while (flag[j] and turn == j) bekle; KRİTİK; flag[i] = 0`. Her satır atomik bir adım.

- `peterson_states(variant)` → `(karşılıklı_dışlama_ihlali_var_mı, kilitlenme_var_mı, durum_sayısı)`; `variant`: `"correct"`, `"turn_first"` (önce `turn = j`, sonra `flag[i] = 1`), `"no_turn"` (yalnız bayraklar: `flag[i] = 1; while flag[j] bekle`).

In [ ]:
%%writefile extra_x2.py
def peterson_states(variant):
    ...


print(peterson_states("correct"))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Okuyucu–yazıcı kilidi ve yazıcı açlığı

Ayrık zamanlı simülasyon: her tikte p_r olasılıkla okuyucu, p_w olasılıkla yazıcı gelir; okuma 3 tik, yazma 5 tik sürer. **Okuyucu öncelikli** kilit: içeride okuyucu varken yeni okuyucular girer. **Yazıcı öncelikli**: bekleyen yazıcı varsa yeni okuyucular bekler.

- `simulate(policy, p_r, p_w, ticks, seed=0)` → `(ortalama yazıcı beklemesi, ortalama okuyucu beklemesi, biten yazıcı sayısı)`.

In [ ]:
%%writefile extra_x3.py
import random


def simulate(policy, p_r, p_w, ticks, seed=0):
    ...


print(simulate("reader", 0.6, 0.05, 5000))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

### E4 · Semaforlarla sınırlı tampon ve zehirli hapla sonlandırma (monitörler)

**Semafor** (Dijkstra, 1965): bir tamsayı sayaç ve bir bekleme kuyruğu. İki atomik işlem:

- $P(s)$ (bekle / *acquire*): sayaç > 0 ise azalt ve devam et; değilse kuyruğa gir ve bloklan.
- $V(s)$ (bildir / *release*): kuyrukta bekleyen varsa ilkini uyandır (izni ona devret, sayaç değişmez); yoksa sayacı artır.

İkili semafor (başlangıç 1) bir kilit gibi kullanılır ama **sahibi yoktur**: herhangi bir thread V yapabilir. Bu yüzden "bir olayı bekle" için de kullanılır (başlangıç 0: biri V yapana kadar P bekler). Sayan semafor (başlangıç n) n özdeş kaynağı sayar. C++20'de `std::counting_semaphore`, `std::binary_semaphore`.

**Sınırlı tampon** üç semaforla: `empty` (boş yuva sayısı, başlangıç kapasite), `full` (dolu yuva, 0), `mutex` (tampona erişim, 1). Sıra kritik: önce kaynağı (`empty`/`full`) bekle, **sonra** mutex'i al.

**Sonlandırma:** tüketiciler `while True` içinde `full` bekler; üreticiler bitince kimse V yapmazsa sonsuza dek beklerler. Çözümlerden biri **zehirli hap** (*poison pill*): üreticilerin bittiğini `done` semaforuyla öğrenen bir kapatıcı, tüketici başına bir tane `None` koyar; `None` alan tüketici çıkar.

Verilen: `Sem`, rastgele zamanlayıcı `run(threads, seed)` — her thread bir üreteçtir ve `("P", s)`, `("V", s)`, `("do", f)` (atomik adım) ister; bloklanan thread'i V uyandırır. Hiç çalışabilir thread kalmamışsa `"deadlock"` döner.

- `sem_P(s, tid)` → `True` (devam) ya da `False` (kuyruğa girdi); `sem_V(s)` → uyandırılan tid ya da `None`.
- `producer(buf, empty, full, mutex, items, done)`, `consumer(buf, empty, full, mutex, out)` (çıktıyı `out`'a ekler, `None` alınca biter), `closer(buf, empty, full, mutex, done, n_prod, n_cons)`.

:::ayrıntı Monitörler: semaforun yapılandırılmış hâli
Monitör (Hoare 1974, Brinch Hansen) = paylaşılan veri + onu koruyan **tek** bir kilit + bir ya da daha çok **koşul değişkeni**; veriye yalnız monitörün yordamlarıyla erişilir. C++'ta `std::mutex` + `std::condition_variable` ile kurulan sınıf, Java'da `synchronized` + `wait/notify`. Semafordan farkı: koşul değişkeninin hafızası yoktur — kimse beklemiyorken yapılan `notify` kaybolur; bu yüzden bekleme her zaman bir **koşul döngüsü** içinde yapılır (`while (!pred) cv.wait(lock)`; sahte uyanmalara da karşı). İkisi eşdeğer güçtedir: monitörle semafor (sayaç + cv), semaforlarla monitör kurulabilir. Uyandırma anlamı: Hoare (uyandırılan hemen çalışır, koşul garanti) ve Mesa (uyandırılan sıraya girer, koşul yeniden denetlenmeli — pratikte hep bu).
:::

:::ayrıntı "call once" ve tembel başlatma
Paylaşılan bir nesneyi ilk kullanımda bir kez oluşturmak: saf "kontrol et, sonra oluştur" yarış koşuludur; "çift denetimli kilitleme" (double-checked locking) bellek sıralaması olmadan bozuktur (yarım kurulmuş nesne görülebilir). C++'ta doğru yollar: `std::call_once` + `std::once_flag`, ya da C++11'den beri thread güvenli olan fonksiyon içi `static` değişken.
:::

In [ ]:
%%writefile extra_x4.py
import random
from collections import deque


class Sem:
    def __init__(self, n):
        self.n = n
        self.q = deque()


def sem_P(s, tid):
    ...


def sem_V(s):
    ...


def run(threads, seed, max_steps=20000):
    rng = random.Random(seed)
    req, blocked, granted = {}, set(), set()
    for i, t in enumerate(threads):
        try:
            req[i] = next(t)
        except StopIteration:
            pass
    steps = 0
    while req:
        runnable = [i for i in req if i not in blocked]
        if not runnable:
            return "deadlock"
        if steps >= max_steps:
            return "timeout"
        i = rng.choice(runnable)
        steps += 1
        op, arg = req[i]
        if op == "P":
            if i in granted:
                granted.discard(i)
            elif not sem_P(arg, i):
                blocked.add(i)
                continue
        elif op == "V":
            w = sem_V(arg)
            if w is not None:
                blocked.discard(w)
                granted.add(w)
        elif op == "do":
            arg()
        try:
            req[i] = threads[i].send(None)
        except StopIteration:
            del req[i]
    return "ok"


def producer(buf, empty, full, mutex, items, done):
    ...


def consumer(buf, empty, full, mutex, out):
    ...


def closer(buf, empty, full, mutex, done, n_prod, n_cons):
    ...

In [ ]:
# Kontrol (E4)
labcheck.run_file("extra_x4.py", CHECKS["x4"])

### E5 · CAS ile kilitsiz yığın ve ABA sorunu (LL/SC, livelock)

**Compare-and-swap** (CAS): `cas(adres, beklenen, yeni)` — bellekteki değer `beklenen`e eşitse `yeni`yi yaz ve `True` döndür, değilse hiçbir şey yapma; tek atomik donanım komutu (x86 `LOCK CMPXCHG`, C++ `compare_exchange_strong/weak`). **LL/SC** (load-link / store-conditional; ARM, RISC-V, POWER): `LL` değeri okur ve adresi izlemeye alır; `SC` ancak araya başka yazma girmediyse yazar. LL/SC değer değil **yazma olayı** izlediği için ABA'ya bağışıktır, ama önbellek satırı olayları yüzünden "sahte başarısızlık" verebilir (bu yüzden `compare_exchange_weak` döngü içinde kullanılır).

**Kilitsiz yığın** (Treiber, 1986): `top`'u oku → yeni düğümün `next`'ini ona bağla → `cas(top, eski, yeni)`; başarısızsa baştan dene. Hiçbir thread kilit tutmadığı için bir thread durdurulsa bile ötekiler ilerler (*lock-free*).

Bellek modeli: `mem["top"] = (işaretçi, etiket)`, `mem["next"][düğüm]`. Her `yield` bir kesilme noktası (zamanlayıcı başka bir thread'e geçebilir). Verilen: `worker`, `schedule`, `stack_ok`.

- `cas(mem, key, expected, new)` — atomik karşılaştır-yaz.
- `push(mem, node, tagged)`, `pop(mem, tagged)` — üreteç; okuma ile CAS arasında `yield`; `pop` çıkardığı düğümü `return` eder (boşsa `None`). `tagged=True` ise başarılı CAS etiketi 1 artırır; `False` ise etiket hep 0 (yalnız işaretçi karşılaştırılır).

:::ayrıntı Livelock ve açlık
**Kilitlenme** (deadlock): herkes bekliyor, kimse çalışmıyor. **Livelock**: herkes çalışıyor ama kimse ilerlemiyor — ör. iki thread aynı anda geri çekilip aynı anda yeniden denediği için CAS'larını sürekli birbirine bozdurması (koridorda karşılaşıp aynı yöne kaçan iki kişi). Çare: rastgele (üstel) geri çekilme. **Açlık** (starvation): sistem ilerliyor ama belirli bir thread hiç sıra alamıyor (E3'teki yazıcı açlığı; burada sürekli CAS'ı kaybeden thread). İlerleme garantileri: *bloklayan* (kilit) ⊂ *engelsiz* (obstruction-free: tek başına çalışan biter) ⊂ *kilitsiz* (lock-free: sistemde biri hep biter) ⊂ *beklemesiz* (wait-free: herkes sınırlı adımda biter).
:::

In [ ]:
%%writefile extra_x5.py
import random


def cas(mem, key, expected, new):
    ...


def push(mem, node, tagged):
    ...


def pop(mem, tagged):
    ...


def worker(mem, rounds, tagged):
    for _ in range(rounds):
        node = yield from pop(mem, tagged)
        if node is not None:
            yield
            yield from push(mem, node, tagged)


def schedule(gens, seed, max_steps=100000):
    rng = random.Random(seed)
    live = list(gens)
    steps = 0
    while live and steps < max_steps:
        g = rng.choice(live)
        try:
            next(g)
        except StopIteration:
            live.remove(g)
        steps += 1
    return steps


def stack_ok(mem, nodes):
    seen, p = [], mem["top"][0]
    while p is not None and len(seen) <= len(nodes):
        seen.append(p)
        p = mem["next"][p]
    return p is None and sorted(seen) == sorted(nodes)

In [ ]:
# Kontrol (E5)
labcheck.run_file("extra_x5.py", CHECKS["x5"])

## Makale ödevi: Boehm (2005): Thread'ler kütüphane olarak gerçeklenemez

[Boehm — Threads Cannot Be Implemented as a Library (PLDI 2005)](https://doi.org/10.1145/1065010.1065042)

Makalenin üç örneğini (eşzamanlı değişken değiştirme, bitişik bit alanlarının yeniden yazılması, kayıt yükseltme/register promotion) C++'ta yeniden üret: `-O0` ve `-O3` ile derle, x86-64 assembly'sini (`objdump -d` ya da godbolt) incele, hatalı davranışı gösteren bir test yaz. Sonra `std::atomic` ve C++11 bellek modeliyle düzelt ve neden düzeldiğini bellek sırası (memory order) kavramlarıyla açıkla.


**Adımlar**

1. d aşamasındaki `racy` modunun neden tanımlı ama yanlış olduğunu tartış.
2. Bit alanı örneğinde iki thread'in komşu alanlara yazmasını test et.

**Kabul ölçütleri**

- Her örnek için hatalı çıktı veren bir çalıştırma (ya da derleyici dönüşümünü gösteren assembly).
- Düzeltilmiş sürümler 10⁶ denemede doğru.

## Makale ödevi: Peterson (1981) ve donanım bellek modeli

[Peterson — Myths About the Mutual Exclusion Problem (IPL 1981)](https://doi.org/10.1016/0020-0190(81)90106-X)

x2'de durum uzayında kanıtladığın Peterson algoritmasını C++'ta (a) düz `int` değişkenlerle, (b) `std::atomic` + `memory_order_relaxed` ile, (c) `memory_order_seq_cst` ile yaz. Kritik bölgede ortak bir sayacı artırıp 10⁷ girişte ihlal say. x86'nın TSO modelinde (a)/(b)'nin neden bozulduğunu (store buffer) açıkla ve `std::atomic_thread_fence` ile düzelt.


**Adımlar**

1. İki thread farklı çekirdeklere sabitlenmeli (`pthread_setaffinity_np`).

**Kabul ölçütleri**

- (a) ve (b)'de ihlal gözlenir, (c)'de gözlenmez.

## Kaynaklar

- [Barlas — Multicore and GPU Programming (2. baskı), Bölüm 3: Threads and concurrency in standard C++](https://www.elsevier.com/books/multicore-and-gpu-programming/barlas/978-0-12-814120-5) — ©
- [Williams — C++ Concurrency in Action (2. baskı)](https://www.manning.com/books/c-plus-plus-concurrency-in-action-second-edition) — ©
- [cppreference — Concurrency support library](https://en.cppreference.com/w/cpp/thread) — CC BY-SA 3.0
- [Boehm — Threads Cannot Be Implemented as a Library (PLDI 2005)](https://doi.org/10.1145/1065010.1065042) — ©
- [Peterson — Myths About the Mutual Exclusion Problem (IPL 1981)](https://doi.org/10.1016/0020-0190(81)90106-X) — ©
- [Coffman, Elphick, Shoshani — System Deadlocks (ACM Computing Surveys 1971)](https://doi.org/10.1145/356586.356588) — ©